In [15]:
import os
#os.chdir("../")
%pwd

'/home/abood/datascienceproject'

In [16]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/machine.data", header=None, names=["vendor","model","MYCT","MMIN","MMAX","CACH","CHMIN","CHMAX","PRP","ERP"])
df = pd.DataFrame(data)
df.head()

,vendor,model,MYCT,MMIN,MMAX,CACH,CHMIN,CHMAX,PRP,ERP
0,adviser,32/60,125,256,6000,256,16,128,198,199
1,amdahl,470v/7,29,8000,32000,32,8,32,269,253
2,amdahl,470v/7a,29,8000,32000,32,8,32,220,253
3,amdahl,470v/7b,29,8000,32000,32,8,32,172,253
4,amdahl,470v/7c,29,8000,16000,32,8,16,132,132


In [17]:
df.isnull().sum()

vendor    0
model     0
MYCT      0
MMIN      0
MMAX      0
CACH      0
CHMIN     0
CHMAX     0
PRP       0
ERP       0
dtype: int64

In [18]:
df.count()

vendor    209
model     209
MYCT      209
MMIN      209
MMAX      209
CACH      209
CHMIN     209
CHMAX     209
PRP       209
ERP       209
dtype: int64

In [19]:
df.shape

(209, 10)

In [20]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: Path
    all_schema: dict

In [21]:
from src.datascience.constants import *
from src.datascience.utils import read_yaml , create_directories, logger

In [22]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])
        
        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            unzip_data_dir = config.unzip_data_dir,
            STATUS_FILE = config.STATUS_FILE,
            all_schema = schema
        )
        return data_validation_config

In [23]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
    
    def validate_all_columns(self):
        try:
            validate_status = None
            data = pd.read_csv(os.path.join(self.config.unzip_data_dir, "machine.data"), header=None, names=["vendor","model","MYCT","MMIN","MMAX","CACH","CHMIN","CHMAX","PRP","ERP"])
            all_cols = list(data.columns)
            all_schema = self.config.all_schema.keys()

            for col in all_cols:
                if col in all_schema:
                    validate_status = False
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"Column: {col} is present in the dataset and matches the schema.\n")
                else:
                    validate_status = False
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"Column: {col} is not present in the dataset and does not match the schema.\n")
            return validate_status
        except Exception as e:
            logger.exception(e)

In [24]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    logger.exception(e)

[2026-05-06 12:52:04,999: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-06 12:52:05,006: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-06 12:52:05,017: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-06 12:52:05,023: INFO: common]: created directory at: artifacts
[2026-05-06 12:52:05,028: INFO: common]: created directory at: artifacts/data_validation
